<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Configuration and secrets

**[Configuration and secrets](https://learning.nextia-ai.com/courses/python/m04/configuration-and-secrets/)** · Python for Practical AI Engineering · Module 4, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** move settings out of the code into environment variables, give each setting a safe default, and keep a secret token out of the code, the files that you share, and the output.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | An internet connection: the setup installs httpx and downloads the data. No account and no real token: the token in this lesson is for a mock service. |
| **You should know** | `fetch_rows` and the choice of source in `count_tickets.py`, from [Basic HTTP client use](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/). |
| **You do not need** | Your local project and its environment. The setup makes the same `ticket-cleaner` folder here, and installs httpx. |
| **Tested** | Python 3.14.6 with httpx 0.28.1, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every example, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/python/m04/configuration-and-secrets/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M04-L03-configuration-and-secrets/configuration-and-secrets-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `httpx` 0.28.1 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `httpx 0.28.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check httpx==0.28.1
import httpx; print("httpx", httpx.__version__)

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It writes the files that you made in the earlier lessons (the code is in the cell). It needs no internet connection.

It also downloads the course's ticket files (`tickets.csv`, `tickets.json`, under 4 KB together) into `data/`, and checks each file's checksum, as you did with `curl` in [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/#get-the-data-file). This part needs an internet connection and no account. If it shows `Could not download`, check your connection and run it again (in Kaggle, turn on **Internet** in the notebook settings). If it shows `wrong checksum`, delete the folder `ticket-cleaner` (in Colab: the **Files** panel on the left), then run it again. The data is made up for this course; it has no real people.

You should see `Ready: ticket-cleaner, 11 files`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

The files are the code of these lessons, as you have it at the end of each one: [Decisions and repetition](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/) (count_tickets.py), [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/) (count_tickets.py as functions), [Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/) (the package `ticket_cleaner` with `report.py`), [Structured records](https://learning.nextia-ai.com/courses/python/m03/structured-records/) (`records.py` and `parsing.py`), [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/) (`files.py`, `split_records` and `build_summary`), [Basic HTTP client use](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/) (`remote.py` and `try_mock.py`).

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson. Download the data files.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/"
DATA = {
    "tickets.csv": "82e3fb4fa6d94a47b9a7b05d940a088596363e4dd11e89bf82ed486bd2325a1a",
    "tickets.json": "a801f505e3d91ff79cefee9c72e53b5f48e983ee152305b7f04e3ac882dd9946",
}

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in DATA.items():
    path = PROJECT / "data" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-cleaner and run again.")


FILES = {
    'count_tickets.py': '''\
from pathlib import Path

from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.remote import fetch_rows
from ticket_cleaner.report import build_summary

SOURCE = "data/tickets.csv"  # a file path, or "remote"
TICKETS_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.json"
OUTPUT_FILE = Path("reports/summary.json")


def load_rows(source: str) -> list[dict]:
    """Return raw records from a file, or from the web service for "remote"."""
    if source == "remote":
        return fetch_rows(TICKETS_URL)
    return read_rows(Path(source))


def main() -> None:
    rows = load_rows(SOURCE)
    tickets, rejected = split_records(rows)
    for record in rejected:
        print(f"Row {record.row} rejected: {'; '.join(record.problems)}")

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    print(f"Report: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()
''',
    'requirements.txt': '''\
httpx==0.28.1
''',
    'ticket_cleaner/__init__.py': '''\
"""Clean support-ticket records and summarise them."""
''',
    'ticket_cleaner/report.py': '''\
from ticket_cleaner.records import Rejected, Ticket


def filter_by_status(tickets: list[Ticket], status: str = "open") -> list[Ticket]:
    """Return the tickets that have the given status."""
    return [ticket for ticket in tickets if ticket.status == status]


def count_by_category(tickets: list[Ticket]) -> dict[str, int]:
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        counts[ticket.category] = counts.get(ticket.category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets: list[Ticket]) -> float | None:
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    return round(sum(ticket.priority for ticket in tickets) / len(tickets), 1)


def build_summary(tickets: list[Ticket], rejected: list[Rejected], status: str) -> dict:
    """Return the report as a dictionary that can be saved as JSON."""
    selected = filter_by_status(tickets, status)
    return {
        "status": status,
        "valid_records": len(tickets),
        "selected": len(selected),
        "by_category": count_by_category(selected),
        "average_priority": average_priority(selected),
        "rejected": [{"row": r.row, "problems": r.problems} for r in rejected],
    }
''',
    'ticket_cleaner/records.py': '''\
from dataclasses import dataclass

STATUSES = {"open", "pending", "closed"}
PRIORITIES = {1, 2, 3}
REQUIRED_FIELDS = ("id", "status", "category", "priority")


@dataclass(frozen=True)
class Ticket:
    """One valid support ticket."""

    id: str
    status: str
    category: str
    priority: int

    def is_urgent(self) -> bool:
        """Return True for priority 1, the most urgent level."""
        return self.priority == 1


@dataclass(frozen=True)
class Rejected:
    """An input record that is not a valid ticket, and why."""

    row: int
    problems: list[str]
''',
    'ticket_cleaner/parsing.py': '''\
from ticket_cleaner.records import PRIORITIES, REQUIRED_FIELDS, STATUSES, Rejected, Ticket


def clean_text(value: object) -> str:
    """Return the value as text with no outer spaces, in small letters."""
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    """Return a list of problems with one raw record. An empty list means valid."""
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")
    return problems


def to_ticket(raw: dict) -> Ticket:
    """Make a Ticket from a raw record that has no problems."""
    return Ticket(
        id=clean_text(raw["id"]).upper(),
        status=clean_text(raw["status"]),
        category=clean_text(raw["category"]),
        priority=int(clean_text(raw["priority"])),
    )


def split_records(rows: list[dict]) -> tuple[list[Ticket], list[Rejected]]:
    """Separate valid tickets from rejected records. Nothing is dropped silently."""
    tickets = []
    rejected = []
    seen_ids = set()
    for row_number, raw in enumerate(rows, start=1):
        problems = find_problems(raw)
        if not problems:
            ticket = to_ticket(raw)
            if ticket.id in seen_ids:
                problems.append(f"duplicate id {ticket.id}")
        if problems:
            rejected.append(Rejected(row=row_number, problems=problems))
        else:
            tickets.append(ticket)
            seen_ids.add(ticket.id)
    return tickets, rejected
''',
    'ticket_cleaner/files.py': '''\
import csv
import json
import os
from pathlib import Path


def read_rows(path: Path) -> list[dict]:
    """Read raw records from a .csv or .json file."""
    if path.suffix == ".csv":
        with open(path, encoding="utf-8", newline="") as file:
            return list(csv.DictReader(file))
    if path.suffix == ".json":
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    raise ValueError(f"cannot read {path.name}: use a .csv or .json file")


def write_json(data: dict, path: Path) -> None:
    """Save data as JSON. Write a temporary file first, so a failure never leaves half a file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
        file.write("\\n")
    os.replace(temporary, path)
''',
    'ticket_cleaner/remote.py': '''\
import httpx


def fetch_rows(url: str, client: httpx.Client | None = None) -> list[dict]:
    """Download raw records as JSON from a ticket service."""
    client = client or httpx.Client(timeout=10.0)
    with client:
        response = client.get(url)
        response.raise_for_status()
        return response.json()
''',
    'try_mock.py': '''\
import httpx

from ticket_cleaner.remote import fetch_rows


def answer_ok(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json=[{"id": "T-2001", "status": "open"}])


def answer_unavailable(request: httpx.Request) -> httpx.Response:
    return httpx.Response(503, text="Service Unavailable")


def main() -> None:
    client = httpx.Client(transport=httpx.MockTransport(answer_ok))
    print(fetch_rows("https://tickets.example/api/tickets", client=client))

    client = httpx.Client(transport=httpx.MockTransport(answer_unavailable))
    fetch_rows("https://tickets.example/api/tickets", client=client)


if __name__ == "__main__":
    main()
''',
}
for name, text in FILES.items():
    (PROJECT / name).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT / name).write_text(text, encoding="utf-8")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.
- `run("count_tickets.py")` runs a file of the project with Python, as `python count_tickets.py` does in your terminal, and shows its output and its exit code. `run("-m", "pytest")` is `python -m pytest`. `env={"NAME": "value"}` sets an environment variable for that one run.
- `fresh("ticket_cleaner.parsing")` imports a module of the project again, so that the notebook uses the newest version of its file. (A plain `import` uses the copy that is already in memory: see [Notebook discipline](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/).)
- The check functions of this lesson's tasks. They print `Check passed.` or `Not yet:` and what is different.

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
import os, sys

def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

import subprocess, sys

def environment(env=None):
    """The environment variables of a run: the notebook's, plus `env` (None
    removes one). No colours, so the output reads as plain text."""
    variables = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
                 "PYTHON_COLORS": "0", "NO_COLOR": "1", "PY_COLORS": "0"}
    variables.pop("FORCE_COLOR", None)
    for name, value in (env or {}).items():
        if value is None:
            variables.pop(name, None)
        else:
            variables[name] = value
    return variables

def run_quiet(*args, env=None):
    """Run `python <args>` in the project folder. Return the result
    (.returncode, .stdout, .stderr)."""
    return subprocess.run([sys.executable, *args], env=environment(env), text=True, capture_output=True)

def run(*args, env=None, streams="both"):
    """Run `python <args>` in the project folder, like the terminal, and show
    the output and the exit code. streams="stdout" or "stderr" shows one only."""
    if streams == "both":  # the two streams together, in the order they were written
        result = subprocess.run([sys.executable, *args], env=environment(env), text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output = result.stdout
    else:
        result = run_quiet(*args, env=env)
        output = result.stdout if streams == "stdout" else result.stderr
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    print(f"(exit code {result.returncode})")

import importlib

def fresh(module_name):
    """Import a module of the project again, from its newest file."""
    for name in list(sys.modules):
        if name == "ticket_cleaner" or name.startswith("ticket_cleaner."):
            del sys.modules[name]
    importlib.invalidate_caches()
    return importlib.import_module(module_name)

from pathlib import Path

def check_try_token():
    """Run try_token.py without and with the token, and compare the results."""
    source = Path("try_token.py").read_text(encoding="utf-8")
    if "fetch_rows(" not in source:
        print("Not yet: try_token.py must call fetch_rows.")
        return
    without = run_quiet("try_token.py", env={"TICKETS_API_TOKEN": None})
    if without.returncode == 0 or "401" not in without.stderr:
        print("Not yet: without TICKETS_API_TOKEN, the script must stop with 401 Unauthorized. "
              "Is the token in the code? Read it from the settings.")
        return
    wrong = run_quiet("try_token.py", env={"TICKETS_API_TOKEN": "wrong-token"})
    if wrong.returncode == 0 or "401" not in wrong.stderr:
        print("Not yet: with a wrong token, the mock service must answer 401.")
        return
    good = run_quiet("try_token.py", env={"TICKETS_API_TOKEN": "test-token-123"})
    if good.returncode or "T-2001" not in good.stdout:
        print("Not yet: with TICKETS_API_TOKEN=test-token-123, the script must print the list of tickets.")
        return
    if "test-token-123" in good.stdout + good.stderr:
        print("Not yet: the output shows the token. Never print a secret.")
        return
    print("Check passed.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Configuration is not code

`count_tickets.py` has two constants that you edit by hand: `SOURCE` and `TICKETS_URL`. They are **configuration**: values that change from one place or one run to the next, while the code stays the same. A token or a password is a **secret**. If configuration is in the code, every change is an edit to the code, and a secret in the code is a secret in the Git history.

## 2. Environment variables

An **environment variable** is a name and a text value that the terminal gives to every program that it starts. In the terminal, `export TICKETS_SOURCE=remote` (macOS and Linux) or `$env:TICKETS_SOURCE = "remote"` (PowerShell) sets one. In this notebook, `run(..., env={"TICKETS_SOURCE": "remote"})` sets it for one run, as `TICKETS_SOURCE=remote python …` does on macOS and Linux.

`os.environ.get(name, default)` reads a variable, with a default when it is not set. `os.environ[name]` stops with `KeyError` instead. Run the next two cells. The first saves a small script that reads the variable both ways. The second runs it without the variable, then with it.

In [ ]:
%%writefile show_source.py
import os

print("get:", os.environ.get("TICKETS_SOURCE", "data/tickets.csv"))
print("[ ]:", os.environ["TICKETS_SOURCE"])

In [ ]:
run("show_source.py", env={"TICKETS_SOURCE": None})
run("show_source.py", env={"TICKETS_SOURCE": "remote"})

> **Check.** The first run shows `get: data/tickets.csv`, then a traceback that ends with `KeyError: 'TICKETS_SOURCE'`, and `(exit code 1)`. The second run shows `get: remote` and `[ ]: remote`. Use `.get()` with a default when there is a safe default. Use `[ ]` only for a setting that the program cannot work without.

## 3. Secrets

Four rules: a secret is **not in the code** (read it from an environment variable), **not in a file that you commit**, **not in the output**, and **if it leaks, replace it** at the service. A web service usually takes a token in the request header `Authorization`, as `Bearer` followed by the token. It answers `401 Unauthorized` if the token is missing or wrong.

## 4. Worked example: a settings module

All the configuration goes in one module, `config.py`. `field(repr=False)` leaves the token out when you print the settings. `os.environ.get("TICKETS_API_TOKEN") or None` makes an empty value count as "no token". There is no default for the token.

Try it in the notebook first. The cell sets two variables in the notebook's own environment (as `export` does in a terminal), and removes them again at the end.

In [ ]:
import os
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Settings:
    source: str
    tickets_url: str
    api_token: str | None = field(repr=False)


def load_settings() -> Settings:
    return Settings(
        source=os.environ.get("TICKETS_SOURCE", "data/tickets.csv"),
        tickets_url=os.environ.get("TICKETS_URL", "https://tickets.example/api"),
        api_token=os.environ.get("TICKETS_API_TOKEN") or None,
    )


print("Defaults:", load_settings())

os.environ["TICKETS_SOURCE"] = "remote"
os.environ["TICKETS_API_TOKEN"] = "test-token-123"

settings = load_settings()
print("Changed: ", settings)
print("Token is set:", settings.api_token is not None)

del os.environ["TICKETS_SOURCE"], os.environ["TICKETS_API_TOKEN"]

> **Check.** You should see `Defaults: Settings(source='data/tickets.csv', tickets_url='https://tickets.example/api')`, then the settings with `source='remote'`, and `Token is set: True`. The token is in the settings, but the printed line does not show it.

> **Your turn: change one thing.** Delete ` = field(repr=False)` from the `api_token` line, and run the cell again. See what a log line would leak. Then put it back.

Now make the real files. Run the next three cells: `config.py`, `remote.py` (`fetch_rows` gets a `token` parameter, and sends the header only when there is a token), and `count_tickets.py`, which gets everything from the settings. `{...} if token else {}` is a **conditional expression**: one value if the condition is true, another one if not.

In [ ]:
%%writefile ticket_cleaner/config.py
import os
from dataclasses import dataclass, field

DEFAULT_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.json"


@dataclass(frozen=True)
class Settings:
    source: str
    tickets_url: str
    api_token: str | None = field(repr=False)


def load_settings() -> Settings:
    """Read the settings from environment variables, with safe defaults."""
    return Settings(
        source=os.environ.get("TICKETS_SOURCE", "data/tickets.csv"),
        tickets_url=os.environ.get("TICKETS_URL", DEFAULT_URL),
        api_token=os.environ.get("TICKETS_API_TOKEN") or None,
    )

In [ ]:
%%writefile ticket_cleaner/remote.py
import httpx


def fetch_rows(
    url: str, token: str | None = None, client: httpx.Client | None = None
) -> list[dict]:
    """Download raw records as JSON from a ticket service."""
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    client = client or httpx.Client(timeout=10.0)
    with client:
        response = client.get(url, headers=headers)
        response.raise_for_status()
        return response.json()

In [ ]:
%%writefile count_tickets.py
from pathlib import Path

from ticket_cleaner.config import Settings, load_settings
from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.remote import fetch_rows
from ticket_cleaner.report import build_summary

OUTPUT_FILE = Path("reports/summary.json")


def load_rows(settings: Settings) -> list[dict]:
    """Return raw records from a file, or from the web service for "remote"."""
    if settings.source == "remote":
        return fetch_rows(settings.tickets_url, token=settings.api_token)
    return read_rows(Path(settings.source))


def main() -> None:
    settings = load_settings()
    print(f"Source: {settings.source}")
    rows = load_rows(settings)
    tickets, rejected = split_records(rows)
    for record in rejected:
        print(f"Row {record.row} rejected: {'; '.join(record.problems)}")

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    print(f"Report: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Document the names of the settings, but not the secret values, in `.env.example`. Run the cell.

In [ ]:
%%writefile .env.example
# Settings for ticket-cleaner. Set them as environment variables.
# TICKETS_SOURCE: a .csv or .json file, or "remote". Default: data/tickets.csv
TICKETS_SOURCE=data/tickets.csv
# TICKETS_URL: the web service for "remote". Default: the course's public file.
TICKETS_URL=
# TICKETS_API_TOKEN: secret. Never write a real token in this file.
TICKETS_API_TOKEN=

## 5. Guided practice: one program, three sources

Run the cell. It runs the script three times: with no `TICKETS_SOURCE`, with the JSON file, and with `remote`.

In [ ]:
run("count_tickets.py", env={"TICKETS_SOURCE": None})
run("count_tickets.py", env={"TICKETS_SOURCE": "data/tickets.json"})
run("count_tickets.py", env={"TICKETS_SOURCE": "remote"})

> **Check.** The first lines are `Source: data/tickets.csv`, `Source: data/tickets.json` and `Source: remote`, and each run ends with `15 rows: 9 valid, 6 rejected.` The source does not change the result.

## 6. Your turn: a service that needs a token

The real ticket service of a company checks a token.

> **Your turn.** Complete `try_token.py` in the next cell:
>
> 1. `mock_service` answers `401` with the text `"missing or wrong token"` when the request does not have the header `Authorization: Bearer test-token-123`, and the tickets when it has it. `request.headers.get("Authorization")` gives the header.
> 2. `main()` reads the settings, makes a client with the mock service, and prints what `fetch_rows` returns. Give it the token from the settings, not from the code.
>
> Run the cell, then the `run` cell (without and with the token), then the check cell.

In [ ]:
%%writefile try_token.py
import httpx

from ticket_cleaner.config import load_settings
from ticket_cleaner.remote import fetch_rows

URL = "https://tickets.example/api/tickets"


def mock_service(request: httpx.Request) -> httpx.Response:
    # Your code here: answer 401 with the text "missing or wrong token"
    # when the header Authorization is not "Bearer test-token-123".
    return httpx.Response(200, json=[{"id": "T-2001", "status": "open"}])


def main() -> None:
    # Your code here: read the settings, make a client with the mock service,
    # and print what fetch_rows returns, with the token from the settings.
    pass


if __name__ == "__main__":
    main()

In [ ]:
run("try_token.py", env={"TICKETS_API_TOKEN": None})
run("try_token.py", env={"TICKETS_API_TOKEN": "test-token-123"})

In [ ]:
check_try_token()

When it is correct, the first run stops with `HTTPStatusError: Client error '401 Unauthorized'`, and the second prints `[{'id': 'T-2001', 'status': 'open'}]`. The token value is in the environment of the run only: not in the code, the output, or a file.

## 7. Failure case: a setting that does not change anything

Tomás sets the source, but the program still reads the CSV file.

> **Predict.** Look at the name of the variable. Then run the cell.

In [ ]:
run("count_tickets.py", env={"TICKET_SOURCE": "remote"})

> **Check.** The first line is `Source: data/tickets.csv`. The variable has a typing mistake: `TICKET_SOURCE` instead of `TICKETS_SOURCE`. In a terminal, the usual cause is a variable that was set in a different terminal window.

**Fix:** show the value in the terminal where you run the program (`echo $TICKETS_SOURCE`, or `$env:TICKETS_SOURCE` in PowerShell). Set it again there, with the exact name from `config.py`. The `Source:` line shows the value that the program really used, which is why `main()` prints it.

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/python/m04/configuration-and-secrets/). They count toward your certificate when you are signed in and enrolled. On your computer, set the variables in your own terminal, as in the guided practice of the lesson.

In this lesson you learned that configuration changes between runs and places, that environment variables belong to one terminal and are read in one settings module, and that a secret is never in the code, a committed file or the output.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Read failures](https://learning.nextia-ai.com/courses/python/m05/read-failures/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m04/configuration-and-secrets/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.